# NFL Coaching Data & Tendencies v2
This notebook creates auditable coaching tables and derives team-game tendencies from play-by-play. It intentionally distinguishes **measured play-by-play features** from **manually sourced coach-tenure facts**. Do not infer historical coordinators, coverage calls, or scheme labels from team outcomes.

Run the main collector first. Set `DATA_ROOT` to its export folder.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd, numpy as np, json

DATA_ROOT=Path("nfl_data")
OUT=DATA_ROOT/"coaching"
OUT.mkdir(parents=True,exist_ok=True)
RUN_TS=datetime.now(timezone.utc).isoformat()

def load_table(stem):
    hits=sorted(list(DATA_ROOT.rglob(f"{stem}.parquet"))+list(DATA_ROOT.rglob(f"{stem}.csv")))
    if not hits:return pd.DataFrame()
    p=hits[0]
    try:return pd.read_parquet(p) if p.suffix==".parquet" else pd.read_csv(p,low_memory=False)
    except Exception as e:print("load failed",p,e);return pd.DataFrame()

def save_table(df,name):
    try:
        p=OUT/f"{name}.parquet";df.to_parquet(p,index=False)
    except Exception:
        p=OUT/f"{name}.csv";df.to_csv(p,index=False)
    print(name,len(df),p)

## 1. Source registry

These are candidate public sources, not a claim that every source has been downloaded. Check current terms, data licenses, and rate limits before use. For factual coach history, keep a URL and a checked date for each row.

In [2]:
sources=pd.DataFrame([
{"source":"nflverse / nflreadpy","purpose":"Play-by-play, schedules, player/team IDs and stats","url":"https://github.com/nflverse/nflverse-data","access":"Public data/package","status":"candidate"},
{"source":"Pro Football Reference","purpose":"Head-coach history and season outcomes for cross-checking","url":"https://www.pro-football-reference.com/","access":"Public website; automated access may be restricted","status":"candidate"},
{"source":"NFL.com","purpose":"Official team and coaching pages","url":"https://www.nfl.com/","access":"Public website","status":"candidate"},
{"source":"Wikidata","purpose":"Coach identity and tenure lead; verify independently","url":"https://www.wikidata.org/","access":"Public structured data","status":"candidate"},
{"source":"Public GitHub repositories","purpose":"Community datasets and mapping candidates","url":"https://github.com/search?q=NFL+coaching+history&type=repositories","access":"Repository-specific license","status":"candidate"},
{"source":"ESPN","purpose":"Team/coach news and change context","url":"https://www.espn.com/nfl/","access":"Public website; endpoints can change","status":"candidate"},
])
sources["catalogued_at_utc"]=RUN_TS
save_table(sources,"coaching_source_registry")
display(sources)

coaching_source_registry 6 nfl_data\coaching\coaching_source_registry.parquet


,source,purpose,url,access,status,catalogued_at_utc
0,nflverse / nflreadpy,"Play-by-play, schedules, player/team IDs and s...",https://github.com/nflverse/nflverse-data,Public data/package,candidate,2026-10-18T08:30:28.356694+00:00
1,Pro Football Reference,Head-coach history and season outcomes for cro...,https://www.pro-football-reference.com/,Public website; automated access may be restri...,candidate,2026-10-18T08:30:28.356694+00:00
2,NFL.com,Official team and coaching pages,https://www.nfl.com/,Public website,candidate,2026-10-18T08:30:28.356694+00:00
3,Wikidata,Coach identity and tenure lead; verify indepen...,https://www.wikidata.org/,Public structured data,candidate,2026-10-18T08:30:28.356694+00:00
4,Public GitHub repositories,Community datasets and mapping candidates,https://github.com/search?q=NFL+coaching+histo...,Repository-specific license,candidate,2026-10-18T08:30:28.356694+00:00
5,ESPN,Team/coach news and change context,https://www.espn.com/nfl/,Public website; endpoints can change,candidate,2026-10-18T08:30:28.356694+00:00


## 2. Coach identity, tenure, changes, and team-season schemas

Populate these from verified source records or merge your own CSVs. One tenure row should represent a coach-role-team interval. Keep the source URL and the date you checked it. Interim status should be explicit.

In [3]:
coaches=pd.DataFrame(columns=["coach_id","coach_name","canonical_name","source_url","source_checked_at"])
coach_tenures=pd.DataFrame(columns=["coach_id","coach_name","role","team","start_date","end_date","start_season","end_season","interim","source_url","source_checked_at"])
coaching_changes=pd.DataFrame(columns=["change_date","season","team","role","coach_out","coach_in","change_type","source_url","source_checked_at"])
team_season_context=pd.DataFrame(columns=["season","team","wins","losses","ties","points_for","points_against","playoff_result","head_coach_id","offensive_coordinator_id","defensive_coordinator_id","source_url"])
# Optional import: put your verified source files in nfl_data/coaching/input/ and load them here.
INPUT=OUT/"input"
if INPUT.exists():
    for filename,dfname in [("coaches.csv","coaches"),("coach_tenures.csv","coach_tenures"),("coaching_changes.csv","coaching_changes"),("team_season_context.csv","team_season_context")]:
        p=INPUT/filename
        if p.exists():
            globals()[dfname]=pd.read_csv(p)
            print("Imported",p,"rows",len(globals()[dfname]))
display(coach_tenures.head())

,coach_id,coach_name,role,team,start_date,end_date,start_season,end_season,interim,source_url,source_checked_at


## 3. Derive team-game tendencies

This code computes team offensive pass/run rates and, if available, mean EPA and success rate. It aggregates actual play-by-play only; it does not label a specific coach as the caller unless a reliable coach-to-game assignment is available.

In [4]:
pbp=load_table("pbp")
team_game=pd.DataFrame()
team_season=pd.DataFrame()
def col(df,opts):return next((c for c in opts if c in df.columns),None)

if not pbp.empty:
    s=col(pbp,["season"]);w=col(pbp,["week"]);g=col(pbp,["game_id"])
    t=col(pbp,["posteam","offense_team","team"]);pt=col(pbp,["play_type"])
    epa=col(pbp,["epa"]);success=col(pbp,["success"])
    if s and t and (pt or col(pbp,["pass_attempt"]) or col(pbp,["rush_attempt"])):
        x=pbp.copy().rename(columns={s:"season",t:"team"})
        if w and w!="week":x=x.rename(columns={w:"week"})
        if g and g!="game_id":x=x.rename(columns={g:"game_id"})
        if pt:
            z=x[pt].astype(str).str.lower()
            x["_pass"]=z.eq("pass").astype(int);x["_rush"]=z.eq("run").astype(int)
        else:
            pc=col(x,["pass_attempt"]);rc=col(x,["rush_attempt"])
            x["_pass"]=pd.to_numeric(x[pc],errors="coerce").fillna(0) if pc else 0
            x["_rush"]=pd.to_numeric(x[rc],errors="coerce").fillna(0) if rc else 0
        x["_play"]=((x["_pass"]+x["_rush"])>0).astype(int)
        if epa:x["_epa"]=pd.to_numeric(x[epa],errors="coerce")
        if success:x["_success"]=pd.to_numeric(x[success],errors="coerce")
        keys=["season","team"]+(["week"] if "week" in x else [])+(["game_id"] if "game_id" in x else [])
        agg={"plays":("_play","sum"),"pass_plays":("_pass","sum"),"rush_plays":("_rush","sum")}
        if "_epa" in x:agg["mean_epa"]=("_epa","mean")
        if "_success" in x:agg["success_rate"]=("_success","mean")
        team_game=x.groupby(keys,dropna=False).agg(**agg).reset_index()
        denom=team_game["plays"].replace(0,np.nan)
        team_game["pass_rate"]=team_game["pass_plays"]/denom
        team_game["rush_rate"]=team_game["rush_plays"]/denom
        team_season=team_game.groupby(["season","team"],dropna=False).agg(
            games=("plays","count"),plays=("plays","sum"),pass_plays=("pass_plays","sum"),rush_plays=("rush_plays","sum"),
            mean_game_pass_rate=("pass_rate","mean"),mean_game_rush_rate=("rush_rate","mean")
        ).reset_index()
        team_season["weighted_pass_rate"]=team_season["pass_plays"]/team_season["plays"].replace(0,np.nan)
        team_season["weighted_rush_rate"]=team_season["rush_plays"]/team_season["plays"].replace(0,np.nan)
    else:print("Could not detect required play-by-play columns; inspect pbp.columns.")
else:print("No pbp table found. Run the main collector first.")

display(team_game.head())
display(team_season.head())

No pbp table found. Run the main collector first.


""


""


## 4. Export and validate

The season aggregates are descriptive summaries. Do **not** join them into earlier weeks of the same season for a historical forecast. Use team-game values lagged by at least one game or compute expanding means from prior games only.

In [5]:
for name,df in {
    "coaches":coaches,"coach_tenures":coach_tenures,"coaching_changes":coaching_changes,
    "team_season_context":team_season_context,"team_game_tendencies":team_game,"team_season_tendencies":team_season
}.items():save_table(df,name)

validation=[]
for name,df in {"coach_tenures":coach_tenures,"coaching_changes":coaching_changes,"team_game_tendencies":team_game,"team_season_tendencies":team_season}.items():
    validation.append({"dataset":name,"rows":len(df),"empty":df.empty,"has_source_url":"source_url" in df.columns,
                       "notes":"Tenure/change records require source URLs; calculated tendency tables derive from play-by-play."})
validation=pd.DataFrame(validation)
display(validation)
validation.to_csv(OUT/"coaching_validation.csv",index=False)

coaches 0 nfl_data\coaching\coaches.parquet
coach_tenures 0 nfl_data\coaching\coach_tenures.parquet
coaching_changes 0 nfl_data\coaching\coaching_changes.parquet
team_season_context 0 nfl_data\coaching\team_season_context.parquet
team_game_tendencies 0 nfl_data\coaching\team_game_tendencies.parquet
team_season_tendencies 0 nfl_data\coaching\team_season_tendencies.parquet


,dataset,rows,empty,has_source_url,notes
0,coach_tenures,0,True,True,Tenure/change records require source URLs; cal...
1,coaching_changes,0,True,True,Tenure/change records require source URLs; cal...
2,team_game_tendencies,0,True,False,Tenure/change records require source URLs; cal...
3,team_season_tendencies,0,True,False,Tenure/change records require source URLs; cal...


## Coaching data that may remain unavailable

Public data often supports head-coach history and team results more reliably than coordinator play-calling assignments or detailed schemes. Coverage for play-action, motion, route concepts, coverage shells, blitz responsibility, and All-22 tracking can be limited or proprietary. Record missingness rather than filling it with guesses.